# Gemma-Titans Training on TPU with Kauldron

This notebook demonstrates the complete pipeline for training and using the **Gemma-Titans** hybrid model on a Google Cloud TPU v5e using the `Kauldron` framework (DeepMind's standard training library).

### Steps included:
1. **Initialization:** Loading base Gemma weights and randomly initializing Titans NLTM modules using `SkipTitans`.
2. **Pre-training (CPT):** Training *only* the Titans memory layers using `kd.optim.partial_updates` to avoid TPU OOM, utilizing a proper dataset.
3. **Save/Load:** Splitting the PyTree to save only the fine-tuned memory weights.
4. **Dialogue Inference:** Running the model and updating the internal memory cache dynamically.

In [1]:
# 0. Environment Setup

# Clone the new kauldron repository
!git clone https://github.com/google-research/kauldron || true
!pip install -q ./kauldron
# Clone the gemma repository if not already present
!git clone https://github.com/google-deepmind/gemma.git || true
!pip install -q ./gemma

# Clone the dialog repository to fix Gemma format issues
!git clone https://github.com/google-deepmind/dialog || true
!pip install -q ./dialog

!pip install -q flax optax seqio



# Ensure our local modules are in the Python path
import sys
import os
sys.path.append(os.getcwd())

Cloning into 'kauldron'...
remote: Enumerating objects: 9546, done.
remote: Counting objects: 100% (153/153), done.
remote: Compressing objects: 100% (130/130), done.
remote: Total 9546 (delta 41), reused 33 (delta 23), pack-reused 9393 (from 3)
Receiving objects: 100% (9546/9546), 2.84 MiB | 9.55 MiB/s, done.
Resolving deltas: 100% (6858/6858), done.
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.8/40.8 kB 4.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.9/46.9 kB 5.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.7/5.7 MB 99.3 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 101.8/101.8 kB 12.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.3/47.3 kB

In [2]:
# %rm -rf /content/Titans_jax

In [3]:
!git clone https://github.com/andrew-veriga/Titans_jax.git
%cd Titans_jax

Cloning into 'Titans_jax'...
remote: Enumerating objects: 250, done.
remote: Counting objects: 100% (36/36), done.
remote: Compressing objects: 100% (25/25), done.
remote: Total 250 (delta 18), reused 25 (delta 11), pack-reused 214 (from 1)
Receiving objects: 100% (250/250), 6.00 MiB | 38.40 MiB/s, done.
Resolving deltas: 100% (148/148), done.
/content/Titans_jax


In [4]:
from kauldron import kd
from kauldron.ktyping import Array

/usr/local/lib/python3.12/dist-packages/jax/_src/cloud_tpu_init.py:86: UserWarning: Transparent hugepages are not enabled. TPU runtime startup and shutdown time should be significantly improved on TPU v5e and newer. If not already set, you may need to enable transparent hugepages in your VM image (sudo sh -c "echo always > /sys/kernel/mm/transparent_hugepage/enabled")
  warnings.warn(


In [5]:
import jax
import jax.numpy as jnp
import optax
from kauldron import kd
import numpy as np
import os
import orbax.checkpoint as ocp

# Gemma imports
from gemma import gm

# Our custom Titans integration
import importlib
import gemma_titans
importlib.reload(gemma_titans)
from gemma_titans import Gemma3_1B_Titans
from titans_ckpts import SkipTitans
import titans_tree_utils

print(f"JAX Backend: {jax.default_backend()}")
print(f"Devices: {jax.devices()}")

# Prevent JAX from allocating 100% of TPU memory instantly
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
# Limit XLA to 85% of TPU HBM to leave room for overhead
os.environ["XLA_PYTHON_CLIENT_MEM_FRACTION"] = ".85"
# Reduce fragmentation and compilation memory spike
os.environ["XLA_FLAGS"] = "--xla_gpu_enable_highest_priority_async_collectives=true --xla_tpu_enable_data_parallel_all_reduce_opt=true --xla_tpu_memory_bound_loop_fusion_limit=1"
os.environ["JAX_COMPILATION_CACHE_DIR"] = "/tmp/jax_cache"


JAX Backend: tpu
Devices: [TpuDevice(id=0, process_index=0, coords=(0,0,0), core_on_chip=0)]


## 1. Model initialization

We load previously trained titans weights and merge them with original Gemma weights

In [ ]:
import shutil
import os

def load_titans_weights(load_dir: str):
    load_path = os.path.abspath(load_dir)
    checkpointer = ocp.StandardCheckpointer()

    # Restore directly without a template (returns dictionary of arrays)
    titans_params = checkpointer.restore(load_path)
    return titans_params

merged_params = None
titans_delta_path = "./saved_titans_delta"
titans_zip = "saved_titans_delta.zip"

if os.path.exists(titans_zip):
    print(f"Unpacking {titans_zip}...")
    shutil.unpack_archive(titans_zip, titans_delta_path)

if os.path.exists(titans_delta_path):
    # Load original Gemma 3 1B IT weights
    print("Loading original Gemma weights...")
    original_params = gm.ckpts.load_params(gm.ckpts.CheckpointPath.GEMMA3_1B_IT)

    # Merge loaded Titans weights into original Gemma params
    print("Merging Titans weights...")
    import titans_tree_utils
    loaded_titans_params = load_titans_weights(titans_delta_path)
    merged_params = titans_tree_utils.merge_titans_params(original_params, loaded_titans_params)
    print("✅ Success: Pre-trained Titans weights loaded and merged.")
else:
    print("⚠️ Note: './saved_titans_delta' not found. Titans layers will be randomly initialized.")

2. Pre-training (CPT)

We use `Kauldron` to orchestrate the training.
- **Dataset:** Instead of a python generator, we use `kd.data.py.Elements`.
- **Model Config:** We use the standard `gm.nn.Gemma3_1B.config`.
- **SkipTitans:** Handles loading Gemma weights while keeping Titans randomized.
- **partial_updates:** Ensures XLA only builds backprop graphs for memory parameters to prevent OOM.

In [6]:
# Define the model configuration
gemma_config = gm.nn.Gemma3_1B.config

# Initialize model
model = Gemma3_1B_Titans(
    config=gemma_config,
    dtype=jnp.bfloat16,
    return_last_only=False,
    tokens="batch.input",
)
model.titans_layer_indices = [11] # Add Titans memory only to the middle layer

# Create a proper Kauldron dataset pipeline using TFDS
batch_size = 16
max_length = 256

def format_squad(x):
    # SQuAD structure: context, question, answers (dict with 'text' list)
    answers = x.get('answers', {}).get('text', [])
    answer = answers[0] if len(answers) > 0 else "Unknown"

    # Robustly handle potential bytes from TFDS
    ctx = x["context"].decode('utf-8') if hasattr(x["context"], 'decode') else x["context"]
    q = x["question"].decode('utf-8') if hasattr(x["question"], 'decode') else x["question"]
    ans = answer.decode('utf-8') if hasattr(answer, 'decode') else answer

    # Create 'src' and 'dst' for the Seq2SeqTask
    x['src'] = "Context: " + ctx + "\n\nQuestion: " + q
    x['dst'] = ans
    return x

tokenizer = gm.text.Gemma3Tokenizer()


In [7]:

def get_train_dataset():
    return kd.data.py.Tfds(
        name="squad",
        split="train",
        shuffle=True,
        num_epochs=10,
        batch_size=batch_size,
        num_workers=4,
        transforms=[
            format_squad,
            gm.data.Seq2SeqTask(
                in_prompt="src",
                in_response="dst",
                out_input="input",
                out_target="target",
                out_target_mask="loss_mask",
                tokenizer=tokenizer,
                max_length=max_length,
                truncate=True,
            ),
            kd.data.py.Elements(keep=["input", "target", "loss_mask"]),
        ],
    )

# Configure Trainer
trainer = kd.train.Trainer(
    seed=42,
    workdir=os.path.abspath('./titans_workdir_squad'),
    train_ds=get_train_dataset(),
    model=model,

    # --- INITIALIZATION LOGIC ---
    # If we have merged_params from Cell 1, use them directly (much faster).
    # Otherwise, use SkipTitans to load Gemma and randomly init Titans.
    init_transform=(lambda state: state.replace(params=merged_params)) if merged_params is not None else SkipTitans(
        wrapped=gm.ckpts.LoadCheckpoint(
            path=gm.ckpts.CheckpointPath.GEMMA3_1B_IT,
        ),
        workdir=os.path.abspath('./SkipTitans_workdir')
    ),

    num_train_steps=10000,
    train_losses={
        "xentropy": kd.losses.SoftmaxCrossEntropyWithIntLabels(
            logits="preds.logits",
            labels="batch.target",
            mask="batch.loss_mask",
        ),
    },

    # --- PREVENT TPU OOM & REDUCE COMPILATION SPIKE ---
    # Using optax.MultiSteps for REAL Gradient Accumulation (Optax 0.2.6)
    optimizer=optax.MultiSteps(
        kd.optim.partial_updates(
            optax.adam(learning_rate=1e-4),
            mask=kd.optim.select(["memory", "memory_gate"]),
        ),
        every_k_schedule=4,
    ),
    checkpointer=kd.ckpts.Checkpointer(save_interval_steps=500),

    # Sharding for TPU / TPU Pods
    sharding=kd.sharding.ShardingStrategy(),
)

print("Trainer initialized. Starting compilation and training...")

# Uncomment to run the actual training loop:
state, aux = trainer.train()

## 3. Save / Load Trained Weights
We don't want to save the entire 1B model, just our new memory projections. We utilize the `split_titans_params` utility.

In [11]:
def save_titans_weights(state: kd.train.TrainState, save_dir: str):
    # 1. Extract only the Titans weights
    _, titans_params = titans_tree_utils.split_titans_params(state.params)

    import shutil
    save_path = os.path.abspath(save_dir)
    if os.path.exists(save_path):
        shutil.rmtree(save_path)

    checkpointer = ocp.StandardCheckpointer()

    # 2. Correct Orbax syntax: save(path, state)
    # Passing titans_params as the positional state saves only them.
    checkpointer.save(save_path, titans_params)
    checkpointer.wait_until_finished()
    print(f"Saved ONLY Titans weights to {save_path}")

def load_titans_weights(load_dir: str):
    load_path = os.path.abspath(load_dir)
    checkpointer = ocp.StandardCheckpointer()

    # Restore directly without a template (returns dictionary of arrays)
    titans_params = checkpointer.restore(load_path)
    return titans_params

# Usage:
save_titans_weights(state, "./saved_titans_delta")
loaded_titans_params = load_titans_weights("./saved_titans_delta")
original_params, _ = titans_tree_utils.split_titans_params(state.params)
state = state.replace(params=titans_tree_utils.merge_titans_params(original_params, loaded_titans_params))

In [9]:
!ls ./saved_titans_delta


array_metadatas       d		      _METADATA        _sharding
_CHECKPOINT_METADATA  manifest.ocdbt  ocdbt.process_0


In [12]:
import os
import zipfile
import shutil
from google.colab import files

checkpoint_dir = "./saved_titans_delta"

# First, let's check if the directory exists
if not os.path.exists(checkpoint_dir):
    print(f"Error: {checkpoint_dir} does not exist!")
else:
    # Try Colab download first
    try:

        # Create a zip file of the checkpoint
        zip_filename = "saved_titans_delta.zip"

        # Remove old zip if exists
        if os.path.exists(zip_filename):
            os.remove(zip_filename)

        # Create zip file
        shutil.make_archive("saved_titans_delta", 'zip', checkpoint_dir)

        # Download the zip file
        files.download(zip_filename)
        print(f"✓ Downloaded {zip_filename} via Colab")

    except ImportError:
        # Not in Colab, create zip and provide manual download instructions
        print("Not running in Google Colab. Creating zip file...")

        zip_filename = "saved_titans_delta.zip"

        # Remove old zip if exists
        if os.path.exists(zip_filename):
            os.remove(zip_filename)

        # Create zip file
        shutil.make_archive("saved_titans_delta", 'zip', checkpoint_dir)

        # Get file size
        file_size = os.path.getsize(zip_filename) / (1024 * 1024)  # Size in MB

        print(f"\n✓ Created {zip_filename} ({file_size:.2f} MB)")
        print(f"📁 Full path: {os.path.abspath(zip_filename)}")
        print("\nTo download, use one of these methods:")
        print("1. Right-click the file in Jupyter file browser and select 'Download'")
        print("2. Use scp/rsync from your terminal:")
        print(f"   scp user@server:{os.path.abspath(zip_filename)} ./")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✓ Downloaded saved_titans_delta.zip via Colab


## 4. Dialogue Inference (Test-Time Dynamic Memory)
During inference, the model passes a `cache` containing both the standard Attention KV-cache AND the Titans Neural Memory State. The model automatically updates its internal state.

In [13]:
importlib.reload(gemma_titans)
# We use Gemma's built-in Sampler to correctly handle positions, attention masks, and cache updates
sampler = gm.text.Sampler(
    model=model,
    params=state.params,
    tokenizer=tokenizer,
)

print("Simulation: User enters a turn...")
prompt = "Context: The Titans are powerful mythological entities.\n\nQuestion: Who are the Titans?"
prompt = "Titans - технология запоминания контекста во время ответа LLM"
# The sampler will automatically prefill the cache, update Titans memory, and generate text
output = sampler.sample(prompt, max_new_tokens=20)

print("\n--- Generated Response ---")
print(output)
print("\nMemory state updated automatically in cache.")

Simulation: User enters a turn...

--- Generated Response ---
 பணி невер.μέtowards 것 nachhائية gầnдин่าpe προς قریبேயהNраз

Memory state updated automatically in cache.


## 5. (Bonus) Fast Loading via `jax.eval_shape`
In a new session, to avoid wasting time and memory initializing random weights before loading, you can create a zero-memory structural template using `jax.eval_shape`.

In [ ]:
import jax
import jax.numpy as jnp
import orbax.checkpoint as ocp
import titans_tree_utils

# 1. Define model config (must match the training config)
model_eval = Gemma3_1B_Titans(
    config=gm.nn.Gemma3_1B.config,
    dtype=jnp.bfloat16,
    return_last_only=False,
    tokens="batch.input",
)
model_eval.titans_layer_indices = [11]

# 2. MAGIC: Get a "hollow" template without allocating HBM memory
template_variables = jax.eval_shape(
    model_eval.init,
    jax.random.PRNGKey(0),
    tokens=jnp.ones((1, 1), dtype=jnp.int32)
)

# 3. Extract the Titans portion of the template
_, titans_template = titans_tree_utils.split_titans_params(template_variables['params'])

# 4. Load the saved weights using this lightweight template
checkpointer = ocp.StandardCheckpointer()
state = load_titans_weights(state, "./saved_titans_delta")
# print("Titans weights loaded successfully!")
sampler = gm.text.Sampler(
    model=model_eval,
    params=state.params,
    tokenizer=tokenizer,
)

print("Simulation: User enters a turn...")
prompt = "Context: The Titans are powerful mythological entities.\n\nQuestion: Who are the Titans?"
prompt = "Titans - технология запоминания контекста во время ответа LLM"
# The sampler will automatically prefill the cache, update Titans memory, and generate text
sampler.sample(prompt, max_new_tokens=20)

Simulation: User enters a turn...


In [ ]:
sampler.sample(prompt, max_new_tokens=20)

' program down.wordzystava text. туда неavut_perwell, text aftertext로,'